# Spectral Joint Flow (SJFPO) — Online Colab / Kaggle (with internet)

Clone the repo with a GitHub token, install requirements, then run training
via `!python scripts/train.py`. No source is embedded in the notebook.

## 0. Secrets and environment

1. Set `GITHUB_TOKEN` in the environment (Colab secrets or Kaggle secrets).
2. The token is read via `load_secret()` using the runtime environment.
3. Run all cells top to bottom.

In [1]:
# Detect runtime and load secrets from the right store.
import os

OUTPUT_DIR = {
    "colab": "/content/",
    "kaggle": "/kaggle/working/",
    "local": "./",
}
INPUT_DIR = {
    "colab": "/content/",
    "kaggle": "/kaggle/input/",
    "local": "./",
}
def detect_env():
    try:
        if "google.colab" in str(get_ipython()):
            return "colab"
    except NameError:
        pass
    if os.environ.get("KAGGLE_KERNEL_RUN_TYPE"):
        return "kaggle"
    return "local"

ENV_NAME = detect_env()
print(f"Environment: {ENV_NAME}")

def load_secret(key_name):
    try:
        if ENV_NAME == "colab":
            from google.colab import userdata
            return userdata.get(key_name) or None
        if ENV_NAME == "kaggle":
            from kaggle_secrets import UserSecretsClient
            return UserSecretsClient().get_secret(key_name) or None
        return os.getenv(key_name)
    except Exception as e:
        print(f"load_secret({key_name}) failed: {e}")
        return None

ENV_NAME = detect_env()


Environment: colab


In [2]:
# Clone the repo using the GitHub token from the environment/store.
%cd $OUTPUT_DIR[ENV_NAME]
!rm -rf spectral-joint-policy
GITHUB_TOKEN = load_secret("GITHUB_TOKEN")
repo_slug = os.getenv("GITHUB_REPO", "dzungphieuluuky/spectral-joint-policy")
repo_url = f"https://github.com/{repo_slug}.git"
if GITHUB_TOKEN:
    repo_url = repo_url.replace("github.com", f"{GITHUB_TOKEN}@github.com")
print(f"Cloning from: {repo_url.replace(GITHUB_TOKEN or '', '<TOKEN>') if GITHUB_TOKEN else repo_url}")
!git clone $repo_url spectral-joint-policy

[Errno 2] No such file or directory: '{colab: /content/, kaggle: /kaggle/working/, local: ./}[ENV_NAME]'
/content
Cloning from: https://<TOKEN>@github.com/dzungphieuluuky/spectral-joint-policy.git
Cloning into 'spectral-joint-policy'...
remote: Enumerating objects: 2266, done.
remote: Counting objects: 100% (143/143), done.
remote: Compressing objects: 100% (85/85), done.
remote: Total 2266 (delta 76), reused 111 (delta 55), pack-reused 2123 (from 1)
Receiving objects: 100% (2266/2266), 1.50 MiB | 16.29 MiB/s, done.
Resolving deltas: 100% (1575/1575), done.


In [3]:
# Move into the cloned repo and run directly (no embedded source).
%cd spectral-joint-policy
!pwd

/content/spectral-joint-policy
/content/spectral-joint-policy


In [4]:
%%writefile requirements-colab.txt
# Deep Learning
torch
torchvision
torchaudio
torchcodec
jax
jaxlib
accelerate
diffusers
datasets
huggingface-hub
safetensors
bitsandbytes
einops
einx

# Reinforcement Learning / Robot Learning
gymnasium
gymnasium-robotics
pettingzoo
mujoco
skrl
lerobot

# Scientific / Vision
numpy
scipy
pandas
matplotlib
scikit-learn
scikit-image
opencv-python-headless
pillow
imageio
imageio-ffmpeg
av
moviepy

# Data / Storage
pyarrow

# Experiment Tracking / Config
wandb
tensorboard
draccus


Overwriting requirements-colab.txt


In [ ]:
# Detect runtime and choose install source.
if ENV_NAME == "colab":
    !uv pip install -r requirements-colab.txt
    # !sudo apt-get update
    # !sudo apt-get install -y pkg-config python3-dev \
    # libavformat-dev libavcodec-dev libavdevice-dev \
    # libavutil-dev libavfilter-dev libswscale-dev libswresample-dev
    !uv pip uninstall av
    !uv pip install "av<15.0.0" --only-binary=:all:
else:
    !uv pip install -r requirements-colab.txt
    # !sudo apt-get update
    # !sudo apt-get install -y pkg-config python3-dev \
    # libavformat-dev libavcodec-dev libavdevice-dev \
    # libavutil-dev libavfilter-dev libswscale-dev libswresample-dev
    !uv pip uninstall av
    !uv pip install "av<15.0.0" --only-binary=:all:


Using Python 3.13.15 environment at: /usr
Checked 36 packages in 110ms
Using Python 3.13.15 environment at: /usr
Uninstalled 1 package in 113ms
 - av==14.2.0
Using Python 3.13.15 environment at: /usr
Resolved 1 package in 4ms
Installed 1 package in 6ms
 + av==14.2.0


In [6]:
# wandb login from secret store (runs before any training cell).
_wandb_key = load_secret("WANDB_API_KEY")
if _wandb_key:
    !wandb login $_wandb_key
else:
    print("WANDB_API_KEY not found; training will run without wandb sync.")

wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: W&B API key is configured. Use `wandb login --relogin` to force relogin


In [ ]:
# Run training directly from the cloned repo.
!SJFPO_COMPILE=0 MPLBACKEND=Agg MUJOCO_GL=egl  python scripts/train.py     --method sjfpo     --env Hopper-v5     --seed 3407     --total_steps 200000     --eval_steps 10     --save_steps 1000     --sjfpo.use_per_bin_clip false --sjfpo.use_dct_transform true

SJFPOPolicy(
  (actor): SJFPOFlowModel(
    (timestep_embedder): SinusoidalEncoding()
    (in_proj): Linear(in_features=3, out_features=128, bias=True)
    (bin_encoding): SinusoidalEncoding()
    (time_mlp): CondMLP(
      (fc1): Linear(in_features=128, out_features=512, bias=True)
      (act): SiLU()
      (fc2): Linear(in_features=512, out_features=128, bias=True)
    )
    (blocks): ModuleList(
      (0-3): 4 x DiTBlock(
        (norm1): RMSNorm((128,), eps=1e-06, elementwise_affine=True)
        (attn): SelfAttention(
          (qkv): Linear(in_features=128, out_features=384, bias=True)
          (out): Linear(in_features=128, out_features=128, bias=True)
        )
        (norm2): RMSNorm((128,), eps=1e-06, elementwise_affine=True)
        (mlp): SwiGLU(
          (gate): Linear(in_features=128, out_features=512, bias=True)
          (up): Linear(in_features=128, out_features=512, bias=True)
          (down): Linear(in_features=512, out_features=128, bias=True)
        )
        